In [0]:
# Path to the S3 bucket
s3_path = "s3a://sppevribucket/"

# Recursive function to list all files in folders and subfolders
def list_files_recursive(path):
    files_and_folders = dbutils.fs.ls(path)
    for item in files_and_folders:
        if item.isDir():
            print(f"Directory: {item.path}")
            list_files_recursive(item.path)  # Recursive call for subdirectories
        else:
            print(f"File: {item.path}")

# List all files and folders recursively
list_files_recursive(s3_path)

Directory: s3a://sppevribucket/output/
File: s3a://sppevribucket/output/H1234567890.json
File: s3a://sppevribucket/output/H1234567891.json
File: s3a://sppevribucket/output/H1234567892.json


In [0]:
from pyspark.sql.functions import *

# Path to the S3 bucket directory
s3_path = "s3a://sppevribucket/output/"

# Configure Auto Loader to read JSON files with schema evolution
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.includeExistingFiles", "true") \
    .option("cloudFiles.schemaLocation", "s3a://sppevribucket/schema-json-demo-location/") \
    .load(s3_path)

# Write the data to a Bronze Delta table with schema evolution and checkpointing
df.writeStream.format("delta") \
    .option("checkpointLocation", "s3a://sppevribucket/checkpoints/bronze_json_demo/") \
    .outputMode("append") \
    .table("eu_west2_space.default.bronze_json_demo")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6769399270693236>, line 13
      4 s3_path = "s3a://sppevribucket/output/"
      6 # Configure Auto Loader to read JSON files with schema evolution
      7 df = spark.readStream.format("cloudFiles") \
      8     .option("cloudFiles.format", "json") \
      9     .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
     10     .option("cloudFiles.inferColumnTypes", "true") \
     11     .option("cloudFiles.includeExistingFiles", "true") \
     12     .option("cloudFiles.schemaLocation", "s3a://sppevribucket/schema-json-demo-location/") \
---> 13     .load(s3_path)
     15 # Write the data to a Bronze Delta table with schema evolution and checkpointing
     16 df.writeStream.format("delta") \
     17     .option("checkpointLocation", "s3a://sppevribucket/checkpoints/bronze_json_demo/") \
     18     .outputMode("app

In [0]:
from pyspark.sql.functions import *

# Path to the S3 bucket directory
s3_path = "s3a://sppevribucket/output/"

# Configure Auto Loader to read JSON files
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.includeExistingFiles", "true") \
    .load(s3_path)

# Display the schema of each file
df.printSchema()


---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-6769399270693237>, line 10
      4 s3_path = "s3a://sppevribucket/output/"
      6 # Configure Auto Loader to read JSON files
      7 df = spark.readStream.format("cloudFiles") \
      8     .option("cloudFiles.format", "json") \
      9     .option("cloudFiles.includeExistingFiles", "true") \
---> 10     .load(s3_path)
     12 # Display the schema of each file
     13 df.printSchema()

File /databricks/spark/python/pyspark/sql/streaming/readwriter.py:307, in DataStreamReader.load(self, path, format, schema, **options)
    302     if type(path) != str or len(path.strip()) == 0:
    303         raise PySparkValueError(
    304             error_class="VALUE_NOT_NON_EMPTY_STR",
    305             message_parameters={"arg_name": "path", "arg_value": str(path)},
    306         )
--> 307     return self._df(self._jreader.load

In [0]:
# Set up Auto Loader for continuous ingestion
(spark.readStream
 .format("cloudFiles")
 .option("cloudFiles.format", "json")
 .option("cloudFiles.schemaLocation", "s3a://sppevribucket/schema-json-demo-location/")  # Stores schema
 .option("cloudFiles.inferColumnTypes", "true")
 .load("s3a://sppevribucket/output/").schema("<your_schema_here>")
 .writeStream
 .option("checkpointLocation", f"s3a://sppevribucket/schema-json-demo-location/_checkpoint")
 .trigger(availableNow=True)  # Change to continuous for real-time
 .toTable("eu_west2_space.default.bronze_json_demo"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6769399270693238>, line 7
      1 # Set up Auto Loader for continuous ingestion
      2 (spark.readStream
      3  .format("cloudFiles")
      4  .option("cloudFiles.format", "json")
      5  .option("cloudFiles.schemaLocation", "s3a://sppevribucket/schema-json-demo-location/")  # Stores schema
      6  .option("cloudFiles.inferColumnTypes", "true")
----> 7  .load("s3a://sppevribucket/output/").schema("<your_schema_here>")
      8  .writeStream
      9  .option("checkpointLocation", f"s3a://sppevribucket/schema-json-demo-location/_checkpoint")
     10  .trigger(availableNow=True)  # Change to continuous for real-time
     11  .toTable("eu_west2_space.default.bronze_json_demo"))

File /databricks/spark/python/pyspark/sql/streaming/readwriter.py:307, in DataStreamReader.load(self, path, format, schema, **options)
    302    

In [0]:
from pyspark.sql.functions import *

# Path to the S3 bucket directory
s3_path = "s3a://sppevribucket/output/"

# Configure Auto Loader to read JSON files
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.includeExistingFiles", "true") \
    .load("s3a://sppevribucket/output/H1234567890.json")

# Display the schema of each file
df.printSchema()


---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-6769399270693239>, line 10
      4 s3_path = "s3a://sppevribucket/output/"
      6 # Configure Auto Loader to read JSON files
      7 df = spark.readStream.format("cloudFiles") \
      8     .option("cloudFiles.format", "json") \
      9     .option("cloudFiles.includeExistingFiles", "true") \
---> 10     .load("s3a://sppevribucket/output/H1234567890.json")
     12 # Display the schema of each file
     13 df.printSchema()

File /databricks/spark/python/pyspark/sql/streaming/readwriter.py:307, in DataStreamReader.load(self, path, format, schema, **options)
    302     if type(path) != str or len(path.strip()) == 0:
    303         raise PySparkValueError(
    304             error_class="VALUE_NOT_NON_EMPTY_STR",
    305             message_parameters={"arg_name": "path", "arg_value": str(path)},
    306         )
--> 307 

In [0]:
from pyspark.sql import SparkSession

# Path to the specific JSON file
file_path = "s3a://sppevribucket/output/H1234567892.json"

# Read the JSON file
df = spark.read.format("json").load(file_path)

# Display the schema of the file
df.printSchema()


root
 |-- _corrupt_record: string (nullable = true)



In [0]:
df

DataFrame[_corrupt_record: string]

In [0]:
# Path to the specific JSON file
file_path = "s3a://sppevribucket/output/H1234567890.json"

# Read the JSON file with corrupt record handling
df = spark.read.option("badRecordsPath", "s3a://sppevribucket/bad-records/").json(file_path)

# Display the schema of the file
df.printSchema()

# Show the data to verify
df.show(truncate=False)


root

++
||
++
++

